INTRINCS
     -Usare delle estensioni del compilatore che espongono alcune
     istruzioni come funzioni. Problema: il codice non è portabile ad altre architetture.
     -Richiedono: #include <immintrin.h> e compilazione con -mavx
          ```c
          __m256 x = _mm256_loadu_ps(&a[i]);   // carica 8 float
          __m256 y = _mm256_loadu_ps(&b[i]);
          __m256 z = _mm256_add_ps(x, y);      // somma 8 float in parallelo
          _mm256_storeu_ps(&c[i], z);          // salva 8 float


In [ ]:
# include < stdio .h >
2 # include < stdlib .h >
3 # include < time .h >
4 # include < immintrin .h >
5
6 float * make_array ( int n)
7 {
8 float * v = ( float *) malloc (n * sizeof ( float ));
9 for ( int i = 0; i < n; i ++) {
10 v[i ] = ( float ) rand () / RAND_MAX ;
11 }
12 return v;
13 }
14
15 int main ( int argc , char * argv [])
16 {
17 const int n = 1 < <20;
18 float * a = make_array (n );
19 float * b = make_array (n );
20 float * c = make_array (n );
21
22 clock_t start = clock () ;
23 for ( int i = 0; i < n; i += 8) {
24 __m256 x = _mm256_loadu_ps (& a [i ]) ;
25 __m256 y = _mm256_loadu_ps (& b [i ]) ;
26 __m256 z = _mm256_add_ps (x , y);
27 _mm256_storeu_ps (& c[ i], z);
28 }
29 clock_t end = clock () ;
30 float ms = ( float ) ( end - start ) / ( CLOCKS_PER_SEC / 1000.0) ;
31 printf (" Somma con istruzioni vettoriali : %f ms\n", ms );
32
33 start = clock () ;
34 for ( int i = 0; i < n; i ++) {
35 c[i ] = a[ i] + b[ i ];
36 }
37 end = clock () ;
38 ms = ( float ) ( end - start ) / ( CLOCKS_PER_SEC / 1000.0) ;
39 printf (" Somma senza istruzioni vettoriali : %f ms\n", ms );
40
41 return 0;
42 }

vettoriali per GCC
    - fornisce estensioni vettoriali generiche
    -sintassi: 
        typedef int v4si __attribute__ ((vector_size (4 * sizeof(int))));

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

typedef int v8si __attribute__ ((vector_size (8 * sizeof(int))));

int sum_vector(int *vec, int len)
{
    int sum = 0;
    for (int i = 0; i < len; i++) {
        sum += vec[i];
    }
    return sum;
}

int sum_vector_simd(int *vec, int len)
{
    v8si sumv = {0, 0, 0, 0, 0, 0, 0, 0};
    int i;
    for (i = 0; i + 7 < len; i += 8) {
        v8si x = *(v8si *)&vec[i];
        sumv += x;
    }

    int sum = sumv[0] + sumv[1] + sumv[2] + sumv[3] + sumv[4] + sumv[5] + sumv[6] + sumv[7];
    for (; i < len; i++) {
        sum += vec[i];
    }
    return sum;
}

float benchmark_scalar_100(int *vec, int len, int *out_sum)
{
    const int runs = 100;
    clock_t start = clock();
    int s = 0;
    for (int r = 0; r < runs; r++) {
        s = sum_vector(vec, len);
    }
    clock_t end = clock();
    if (out_sum != NULL) {
        *out_sum = s;
    }
    return (float)(end - start) / (CLOCKS_PER_SEC / 1000.0f) / runs;
}

float benchmark_simd_100(int *vec, int len, int *out_sum)
{
    const int runs = 100;
    clock_t start = clock();
    int s = 0;
    for (int r = 0; r < runs; r++) {
        s = sum_vector_simd(vec, len);
    }
    clock_t end = clock();
    if (out_sum != NULL) {
        *out_sum = s;
    }
    return (float)(end - start) / (CLOCKS_PER_SEC / 1000.0f) / runs;
}

#define N 1000000

int main(int argc, char *argv[])
{
    int v[N];
    for (int i = 0; i < N; i++) {
        v[i] = rand() % 100;
    }

    int s = 0;
    int sv = 0;

    float ms_scalar = benchmark_scalar_100(v, N, &s);
    float ms_simd = benchmark_simd_100(v, N, &sv);

    printf("scalar avg (100 run): %f ms\n", ms_scalar);
    printf("simd avg (100 run): %f ms\n", ms_simd);
    printf("s = %d, sv = %d\n", s, sv);

    return 0;
}

Multithreading con OpenMP
    -api supporto programmazione parallerla in sistemi multi processore e memoria condivisa
    -viene usato dai thread
        -processori eseguono codice parallelo, accedono stessa memoria
    -tipologie: processori ma memoria non condivisa

    -struttura: thread princiaple parte secondari, distribuzione lavoro thread

    -utilizzo: importare omp.h, compilare con -fopenmp(su macos devi usare clang con llvm-openmp)
    -pragma #pragma omp parallel{ codice eseguito tutti thread}
        int x = 2; // ogni thread ha sua copia di x, inizializzata a 2
    -fare cose distinte thread
        -ottenimento numero thread fatto dentro e non fuori, posso specificare numro thread eseguire ses parallela es: 
    

In [11]:
//% cflags: -Xpreprocessor -fopenmp -I/opt/homebrew/opt/libomp/include
//% ldflags: -L/opt/homebrew/opt/libomp/lib -lomp
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[])
{
    printf("a\n");
    #pragma omp parallel
    {
        printf("b\n");
        printf("c\n");
    }
    printf("d\n");
    return 0;
}

a
b
c
b
c
b
c
b
c
b
c
b
c
b
c
b
c
d


In [10]:
//% cflags: -Xpreprocessor -fopenmp -I/opt/homebrew/opt/libomp/include
//% ldflags: -L/opt/homebrew/opt/libomp/lib -lomp
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[])
{
    const int n = 1 << 20;
    int *v = (int *)malloc(n * sizeof(int));
    for (int i = 0; i < n; i++) {
        v[i] = rand() % 100;
    }

    int sum = 0;
    #pragma omp parallel
    {
        int thread_id = omp_get_thread_num();
        int n_threads = omp_get_num_threads();
        int slice_size = n / n_threads;
        int start = thread_id * slice_size;
        int end = (thread_id == n_threads - 1) ? n : (thread_id + 1) * slice_size;

        int partial_sum = 0;
        for (int i = start; i < end; i++) {
            partial_sum += v[i];
        }

        sum += partial_sum;

        printf("Thread %d partial sum: %d\n", thread_id, partial_sum);
    }

    printf("Total sum: %d\n", sum);
    free(v);
    return 0;
}

Thread 4 partial sum: 6490424
Thread 0 partial sum: 6494258
Thread 3 partial sum: 6480228
Thread 2 partial sum: 6481357
Thread 7 partial sum: 6482043
Thread 6 partial sum: 6505154
Thread 5 partial sum: 6469489
Thread 1 partial sum: 6480114
Total sum: 45402839


Sincronizzazione
    Sezioni critiche
        - tutti i thread eseguono, ma solo una alla volta (protetta da un lock)
        - lock preso inizio sezione critica, rilascia fine
    parallel for
        -paralizza cicli for, ogni iterazione opera variabili differenti
        #pragma omp parallel for
            for(int i=0; i<16; i++){
                f(i);
            }
        nowait : aspettiamo tutti finiscano, nowait non aspettiamo
    schedule: possiamo decidere come speszzzati i thread tra cicli
        -default assegna prima iterazione primo thread, seconda secondo, ecc 
        -possiamo decidere cambiare dimensione k, k primo, k secondo, ecc
        -dynamic: spezziamo iterazioni, quando finisco prendo successivo (use case: iterazioni dimensioni diverse), più costosa -> trovare giusto trade off



In [5]:
//% cflags: -Xpreprocessor -fopenmp -I/opt/homebrew/opt/libomp/include
//% ldflags: -L/opt/homebrew/opt/libomp/lib -lomp
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[])
{
    const int n = 1 << 24;
    int *v = (int *)malloc(n * sizeof(int));
    sum = 0;
    for (int i = 0; i < n; i++) {
        v[i] = rand() % 10;
        sum += v[i];
    }
    printf("Total sum: %d\n", sum);
  #pragma omp parallel for
    for (int i = 0; i < n; i++) {
        v[i] = v[i] * 2+1;
    }
    sum = 0;
    printf("Total sum: %d\n", sum);
    return 0;

  #pragma omp parallel for
    for (int i = 0; i < 20; i++) {
        printf("Iterazione = %d\n", i);
    }
    return 0;
}


/var/folders/8j/d85tjxzx213g5grv_btb77500000gn/T/tmpyfxvi07z.c:11:5: error: use of undeclared identifier 'sum'
   11 |     sum = 0;
      |     ^
/var/folders/8j/d85tjxzx213g5grv_btb77500000gn/T/tmpyfxvi07z.c:14:9: error: use of undeclared identifier 'sum'
   14 |         sum += v[i];
      |         ^
/var/folders/8j/d85tjxzx213g5grv_btb77500000gn/T/tmpyfxvi07z.c:16:31: error: use of undeclared identifier 'sum'
   16 |     printf("Total sum: %d\n", sum);
      |                               ^
/var/folders/8j/d85tjxzx213g5grv_btb77500000gn/T/tmpyfxvi07z.c:21:5: error: use of undeclared identifier 'sum'
   21 |     sum = 0;
      |     ^
/var/folders/8j/d85tjxzx213g5grv_btb77500000gn/T/tmpyfxvi07z.c:22:31: error: use of undeclared identifier 'sum'
   22 |     printf("Total sum: %d\n", sum);
      |                               ^
5 errors generated.
[C kernel] GCC exited with code 1, the executable will not be executed

In [6]:
//% cflags: -Xpreprocessor -fopenmp -I/opt/homebrew/opt/libomp/include
//% ldflags: -L/opt/homebrew/opt/libomp/lib -lomp
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[])
{
    const int n = 1 << 24;
    int *v = (int *)malloc(n * sizeof(int));
    int sum = 0;
    for (int i = 0; i < n; i++) {
        v[i] = rand() % 10;
        sum += v[i];
    }
    printf("Total sum before: %d\n", sum);
    
    #pragma omp parallel for
    for (int i = 0; i < n; i++) {
        v[i] = v[i] * 2 + 1;
    }
    
    sum = 0;
    for (int i = 0; i < n; i++) {
        sum += v[i];
    }
    printf("Total sum after: %d\n", sum);
    
    #pragma omp parallel for
    for (int i = 0; i < 20; i++) {
        printf("Iterazione = %d\n", i);
    }
    
    free(v);
    return 0;
}

Total sum before: 75509508
Total sum after: 167796232
Iterazione = 0
Iterazione = 1
Iterazione = 2
Iterazione = 12
Iterazione = 13
Iterazione = 9
Iterazione = 10
Iterazione = 11
Iterazione = 6
Iterazione = 7
Iterazione = 8
Iterazione = 3
Iterazione = 4
Iterazione = 5
Iterazione = 18
Iterazione = 19
Iterazione = 16
Iterazione = 17
Iterazione = 14
Iterazione = 15
